In [18]:
from pathlib import Path
import os
import shutil
import SimpleITK as sitk
from skimage import measure
from scipy import ndimage
import json
from scipy.ndimage import label
import numpy as np


def seg_to_instances(
    seg: np.ndarray,
    min_num_voxel: int = 0,
):
    """
    Use connected components with ones matrix to created
    instances from segmentation

    Args:
        seg: semantic segmentation [spatial dims]
        min_num_voxel: minimum number of voxels of an instance

    Returns:
        np.ndarray: instance segmentation
        Dict[int, int]: mapping from instances to classes
    """
    structure = np.ones([3] * seg.ndim)

    unique_classes = np.unique(seg)
    unique_classes = unique_classes[unique_classes > 0]

    instances = np.zeros_like(seg)
    instance_classes = {}

    i = 1
    for uc in unique_classes:
        binary_class_mask = seg == uc
        instances_temp, _ = label(binary_class_mask, structure=structure)

        instance_ids = np.unique(instances_temp)
        instance_ids = instance_ids[instance_ids > 0]

        for iid in instance_ids:
            instance_binary_mask = instances_temp == iid

            if min_num_voxel > 0:
                if instance_binary_mask.sum() < min_num_voxel:  # remove small instances
                    continue

            instances[instance_binary_mask] = i  # save instance to final mask
            instance_classes[int(i)] = uc
            i = i + 1  # bump instance index
    return instances, instance_classes


path_train = Path("/data/aneurysm/internal_train/crop_0.4")
path_label_trian = Path("/data/aneurysm/internal_train/crop_0.4_label")

path_train_tgt = Path("/data/det_data/Task100_Aneurysm/raw_splitted/imagesTr")
path_label_train_tgt = Path("/data/det_data/Task100_Aneurysm/raw_splitted/labelsTr")


def map_fun_train(x, label=False):
    name = str(x.name)
    case_num = int(name.split(".")[0][2:]) - 1
    if not label:
        nnet_name = f"case_{case_num:04d}_0000.nii.gz"
    else:
        nnet_name = f"case_{case_num:04d}.nii.gz"
    return name, nnet_name


import tqdm

In [ ]:
# get all files in path train




map_cases_train = {}

for x in path_train.iterdir():
    name, nnet_name = map_fun_train(x)

    map_cases_train[name] = nnet_name
    # create a copy of the file in path_train_tgt
    tgt = path_train_tgt / nnet_name
    shutil.copy(x, tgt)

In [23]:
np.unique(instances)

array([0, 1], dtype=uint16)

In [ ]:
# read all files in path_label_train

label_files = list(path_label_trian.iterdir())

# for each file

for x in tqdm.tqdm(label_files):
    header = sitk.ReadImage(str(x))
    # get array
    array = sitk.GetArrayFromImage(header)
    name, nnet_name = map_fun_train(x, label=True)
    tgt = path_label_train_tgt / nnet_name

    # labeled_array, num_features = label(
    #    array, structure=ndimage.generate_binary_structure(3, 3)
    # )
    instances, instance_classes = seg_to_instances(array)
    # create new header from instances
    header_new = sitk.GetImageFromArray(instances)
    header_new.CopyInformation(header)
    sitk.WriteImage(header_new, str(tgt))
    # print(instance_classes)

    # turn dict keys into strings
    instance_classes = {str(k): int(v) - 1 for k, v in instance_classes.items()}
    json_name = path_label_train_tgt / f"{nnet_name[:-7]}.json"
    dict_json = {"instances": instance_classes}
    # save dict as json
    with open(str(json_name), "w") as f:
        json.dump(dict_json, f)
    #    json.dump(dict_json, f)

 16%|█▌        | 186/1186 [15:14<1:36:16,  5.78s/it]

In [54]:
# read all json files in path_label_test_tgt and rewrite them with double quotes instead of single quotes
import json

for x in path_label_train_tgt.iterdir():

    if x.suffix == ".json":
        txt = x.read_text()
        txt = eval(txt)
        with open(str(x), "w") as outfile:
            json.dump(txt, outfile)

In [ ]:
# read all files i

In [7]:
path_test = Path("/data/aneurysm/internal_test/crop_0.4")
path_label_test = Path("/data/aneurysm/internal_test/crop_0.4_label")

path_test_tgt = Path("/data/det_data/Task100_Aneurysm/raw_splitted/imagesTs")
path_label_test_tgt = Path("/data/det_data/Task100_Aneurysm/raw_splitted/labelsTs")

# get all files in path test


def map_fun_test(x, label=False):
    name = str(x.name)
    case_num = int(name.split(".")[0][2:]) - 1
    if not label:
        nnet_name = f"case_{case_num:04d}_0000.nii.gz"
    else:
        nnet_name = f"case_{case_num:04d}.nii.gz"

    return name, nnet_name

In [ ]:


map_cases_test = {}

for x in path_test.iterdir():
    name, nnet_name = map_fun_test(x)

    map_cases_test[name] = nnet_name
    # create a copy of the file in path_train_tgt
    tgt = path_test_tgt / nnet_name
    shutil.copy(x, tgt)

In [17]:
label_files = list(path_label_test.iterdir())

# for each file

for x in tqdm.tqdm(label_files):
    header = sitk.ReadImage(str(x))
    # get array
    array = sitk.GetArrayFromImage(header)
    name, nnet_name = map_fun_test(x, label=True)
    tgt = path_label_test_tgt / nnet_name
    # shutil.copy(x, tgt)
    # labeled_array, num_features = label(
    #    array, structure=ndimage.generate_binary_structure(3, 3)
    # )
    instances, instance_classes = seg_to_instances(array)
    # print(instance_classes)

    json_name = path_label_test_tgt / f"{nnet_name[:-7]}.json"
    # turn dict keys into strings
    instance_classes = {str(k): int(v) for k, v in instance_classes.items()}
    # dict_json = {"instances": {f"{i}": 0 for i in range(1, num_features + 1)}}
    with open(str(json_name), "w") as f:
        json.dump(instance_classes, f)
    #    json.dump(dict_json, f)

    # save dict as json

100%|██████████| 152/152 [11:14<00:00,  4.44s/it]


In [49]:
txt["instances"]

{'1': 0}